# Task 1 Kaggle Train Notebook (`google/gemma-4-E4B-it`)

Notebook nay dung cho huong train nhe hon tren Kaggle voi `Gemma 4 E4B-it`.

Luong chay:
1. clone repo + setup env
2. restart kernel
3. login Hugging Face
4. sanity check model/tokenizer
5. download `AvaMERG + ESConv`
6. dump prompt
7. smoke test
8. train that
9. upload checkpoint len Hugging Face Hub

> Muc tieu cua notebook nay la `text-only SFT`, nen minh uu tien duong `AutoTokenizer` + `run_training(...)` on dinh hon duong multimodal day du.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/QuangVoAI/multimodal-empathy-mental-health.git"
REPO_DIR = "/kaggle/working/multimodal-empathy-mental-health"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Giu torch/CUDA co san cua Kaggle, chi cap nhat stack Hugging Face + BNB.
%pip uninstall -y datasets transformers huggingface_hub accelerate peft bitsandbytes sentencepiece tokenizers torchvision
%pip install --no-cache-dir --force-reinstall \
  "accelerate>=1.7.0" \
  "datasets>=3.6.0" \
  "peft>=0.15.2" \
  "sentencepiece>=0.2.0" \
  "huggingface_hub>=0.34.0,<1.0" \
  "safetensors>=0.6.0" \
  "tqdm>=4.67.0" \
  tokenizers \
  bitsandbytes==0.48.2
%pip install --no-cache-dir git+https://github.com/huggingface/transformers.git


## Restart kernel now

Sau khi cell install chay xong, hay **Restart Session / Restart Kernel** roi moi chay tiep.


In [ ]:
%cd /kaggle/working/multimodal-empathy-mental-health
import torch, transformers, huggingface_hub, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("peft:", peft.__version__)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
MODEL_ID = "google/gemma-4-E4B-it"
RUN_NAME = "task1_gemma4_e4b_it_kaggle"
HF_MODEL_REPO_ID = "SpringWang08/multimodal-empathy-mental-health-gemma4-e4b-task1"

login(HF_TOKEN, add_to_git_credential=False)
print("HF login ok")


In [ ]:
from transformers import AutoProcessor, AutoTokenizer

processor = None
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    print("Processor loaded:", processor.__class__.__name__)
except Exception as e:
    print("AutoProcessor unavailable in current environment:", repr(e))
    print("Falling back to tokenizer-only path for text-only SFT.")

tokenizer = getattr(processor, "tokenizer", None)
if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval

from pathlib import Path
ava_path = Path("data/raw/avamerg/train.json")
esc_path = Path("data/raw/esconv/ESConv.json")
print("AvaMERG file exists:", ava_path.exists(), ava_path)
print("ESConv file exists:", esc_path.exists(), esc_path)
if not ava_path.exists() or not esc_path.exists():
    raise FileNotFoundError("Dataset download chua xong hoac bi fail. Kiem tra HF login va output cua cell download.")


In [ ]:
import json
from pathlib import Path

ava = json.loads(Path("data/raw/avamerg/train.json").read_text(encoding="utf-8"))
esc = json.loads(Path("data/raw/esconv/ESConv.json").read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("ESConv dialogues:", len(esc))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_gemma4_e4b",
    max_length=384,
    max_response_tokens=64,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1.0,
    logging_steps=5,
    save_steps=100,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
    max_train_samples=8,
    gradient_checkpointing=True,
)

run_training(Namespace(**base_cfg))


In [ ]:
!sed -n '1,200p' outputs/sft/debug_gemma4_e4b/example_prompts.json


In [ ]:
smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/gemma4_e4b_smoke",
    "dump_example_prompts": False,
    "max_train_samples": 8,
    "max_length": 384,
    "max_response_tokens": 64,
    "gradient_accumulation_steps": 1,
    "max_steps": 1,
    "logging_steps": 1,
})

run_training(Namespace(**smoke_cfg))


In [ ]:
# Chi chay cell nay sau khi smoke test da qua.
train_cfg = dict(base_cfg)
train_cfg.update({
    "output_dir": f"outputs/sft/{RUN_NAME}",
    "dump_example_prompts": False,
    "max_train_samples": None,
    "max_length": 384,
    "max_response_tokens": 64,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "logging_steps": 10,
    "save_steps": 100,
    "max_steps": -1,
})

run_training(Namespace(**train_cfg))


In [ ]:
from pathlib import Path

final_dir = Path(f"outputs/sft/{RUN_NAME}/final")
print("Final checkpoint dir:", final_dir)
print("Exists:", final_dir.exists())
if final_dir.exists():
    for path in sorted(final_dir.iterdir()):
        print(path.name)


In [ ]:
from scripts.publish_to_hub import create_repo, upload_folder

create_repo(repo_id=HF_MODEL_REPO_ID, repo_type="model", private=False, exist_ok=True)
upload_folder(
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    folder_path=str(final_dir),
    commit_message=f"Upload {RUN_NAME} checkpoint",
)
print(f"Uploaded {final_dir} -> https://huggingface.co/{HF_MODEL_REPO_ID}")
